In [1]:
import sys, json
from pathlib import Path
import numpy as np
import pandas as pd
from galois import GF2
from randextract import ToeplitzHashing

PROJECT_DIR = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(PROJECT_DIR))
sys.path.insert(0, str(PROJECT_DIR / 'scripts'))

import deploy_fabric as deploy
from qne.config import ScenarioConfig
from qne.cascade import Key, ORIGINAL, Reconciliation, MockClassicalSession
from qne.cascade.fault_injection import SDCFaultInjector
from qne.cascade.key import key_from_sifted_json
from qne.cascade.sweep_utils import (
    run_condition_sweep, summarize_outcomes,
    run_real_channel_trial, run_real_channel_reconciliation_trial, collect_key_pairs,
)

print("Setup complete")

Setup complete


In [2]:
SLICE_NAME = 'qfabric-bb84-2'
SCENARIO = 'validation/scenarios/fabric_1km.yml'

fablib = deploy.get_fablib()
slice_obj = fablib.get_slice(name=SLICE_NAME)
slice_obj.show()

alice = slice_obj.get_node("alice")
bob = slice_obj.get_node("bob")
bob_ip = "10.10.1.2"

alice_mac = alice.get_interface(network_name="net_alice_switch").get_mac()
bob_mac = bob.get_interface(network_name="net_switch_bob").get_mac()
sw_alice_mac = slice_obj.get_node("switch").get_interface(network_name="net_alice_switch").get_mac()

Orchestrator,orchestrator.fabric-testbed.net
Credential Manager,cm.fabric-testbed.net
Core API,uis.fabric-testbed.net
Artifact Manager,artifacts.fabric-testbed.net
CEPH Manager,https://ceph-mgr.fabric-testbed.net
Token File,/home/fabric/work/fabric_config/id_token.json
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
Bastion Host,bastion.fabric-testbed.net
Bastion Username,audreyf_0000527467
Bastion Private Key File,/home/fabric/work/fabric_config/fabric_bastion_key
Slice Public Key File,/home/fabric/work/fabric_config/slice_key.pub


User: audreyf@illinois.edu bastion key is valid!
Configuration is valid


ID,900e9b70-47f3-4394-a831-e522acb47878
Name,qfabric-bb84-2
Lease Expiration (UTC),2026-08-25 21:17:14 +0000
Lease Start (UTC),2026-08-11 21:17:14 +0000
Project ID,24f4c8f3-e872-492a-9a83-b48211a91966
State,StableOK
Email,audreyf@illinois.edu
UserId,8616dd8c-5b61-45db-84bb-791c21e82a89


In [41]:
from qne.cascade.parameter_estimation import split_pe_and_generation
import json as _json

RAW_ALICE = PROJECT_DIR / "results" / "fabric_alice_sifted_bits.json"
RAW_BOB = PROJECT_DIR / "results" / "fabric_bob_sifted_bits.json"
GEN_ALICE = PROJECT_DIR / "results" / "fabric_alice_sifted_bits_genonly.json"
GEN_BOB = PROJECT_DIR / "results" / "fabric_bob_sifted_bits_genonly.json"
META_PATH = PROJECT_DIR / "results" / "fabric_key0_pe_meta.json"

alice_key_full, alice_indices_full = key_from_sifted_json(str(RAW_ALICE), "alice_bits")
bob_key_full, bob_indices_full = key_from_sifted_json(str(RAW_BOB), "bob_bits")
assert alice_indices_full == bob_indices_full, "alice and bob's matching indices don't match"

if GEN_ALICE.exists() and GEN_BOB.exists() and META_PATH.exists():
    # Already split in a previous run of this notebook -- reuse it rather
    # than re-splitting (re-splitting an already-split key would sample a
    # PE subset from what's actually generation-only bits, which is wrong).
    alice_key, alice_indices = key_from_sifted_json(str(GEN_ALICE), "alice_bits")
    bob_key, bob_indices = key_from_sifted_json(str(GEN_BOB), "bob_bits")
    meta = _json.loads(META_PATH.read_text())
    k_pe, real_qber = meta["k"], meta["qber"]
    print(f"Reusing existing PE split: k={k_pe}, real_qber={real_qber:.4f}")
else:
    split = split_pe_and_generation(alice_key_full, bob_key_full, sample_fraction=0.1, seed=999001)
    alice_key, bob_key = split["alice_gen"], split["bob_gen"]
    k_pe, real_qber = split["k"], split["qber"]

    gen_indices = split["gen_indices"]
    gen_matching_indices = [alice_indices_full[idx] for idx in gen_indices]
    GEN_ALICE.write_text(_json.dumps({"alice_bits": alice_key.bits.tolist(), "matching_indices": gen_matching_indices}))
    GEN_BOB.write_text(_json.dumps({"bob_bits": bob_key.bits.tolist(), "matching_indices": gen_matching_indices}))
    META_PATH.write_text(_json.dumps({"k": k_pe, "qber": real_qber, "m": split["m"], "n": split["n"]}))

    print(f"m={split['m']} sifted, k={k_pe} PE sample, n={split['n']} generation bits, "
          f"real_qber (from disjoint PE sample) = {real_qber:.4f}")

# Re-sync both remote nodes to this exact (PE-split, generation-only) key pair.
alice.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_ALICE), "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(GEN_BOB), "qfabric/results/bob_sifted_bits.json")
print("Remote nodes synced to PE-split generation-only key")


m=3876 sifted, k=388 PE sample, n=3488 generation bits, real_qber (from disjoint PE sample) = 0.0077
Remote nodes synced to PE-split generation-only key


In [42]:
conditions = [
    ("baseline", {}),
    ("toeplitz_only", {"toeplitz_prob": 0.5}),
    ("final_key_only", {"final_key_prob": 0.5}),
    ("reconciliation_only", {"reconciliation_prob": 0.1}),
    ("combined", {"toeplitz_prob": 0.5, "final_key_prob": 0.5, "reconciliation_prob": 0.1}),
]

all_dfs, summary_rows = {}, []
for label, kwargs in conditions:
    print(f"\nRunning {label}...")
    df_cond = run_condition_sweep(alice_key, bob_key, real_qber, label, n_runs=10, **kwargs)
    all_dfs[label] = df_cond
    summary_rows.append(summarize_outcomes(df_cond, label))

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

full_df = pd.concat(all_dfs.values(), ignore_index=True)
full_df.to_csv(str(PROJECT_DIR / "results" / "sdc_real_key_results.csv"), index=False)



Running baseline...
  run 0: non_convergent=False, keys_match=True
  run 1: non_convergent=False, keys_match=True
  run 2: non_convergent=False, keys_match=True
  run 3: non_convergent=False, keys_match=True
  run 4: non_convergent=False, keys_match=True
  run 5: non_convergent=False, keys_match=True
  run 6: non_convergent=False, keys_match=True
  run 7: non_convergent=False, keys_match=True
  run 8: non_convergent=False, keys_match=True
  run 9: non_convergent=False, keys_match=True

=== baseline (n=10) ===
  Non-convergent:            0/10
  Converged, matched:        10/10
  Converged, mismatched:     0/10
  Converged, unmeasurable:   0/10
  Mismatched, UNDETECTED by verification: 0/10  <-- silent corruption
  Mismatched, detected by verification:   0/10

Running toeplitz_only...
  run 0: non_convergent=False, keys_match=False
  run 1: non_convergent=False, keys_match=True
  run 2: non_convergent=False, keys_match=True
  run 3: non_convergent=False, keys_match=True
  run 4: non_co

In [ ]:
recon_probs = [0.3, 0.1, 0.03, 0.01, 0.003, 0.001]
recon_dfs, recon_summary_rows = [], []

for prob in recon_probs:
    label = f"reconciliation_real_{prob}"
    df_cond = run_condition_sweep(alice_key, bob_key, real_qber, label, n_runs=10, reconciliation_prob=prob)
    recon_dfs.append(df_cond)
    recon_summary_rows.append(summarize_outcomes(df_cond, label))

recon_summary_df = pd.DataFrame(recon_summary_rows)
print(recon_summary_df.to_string(index=False))

recon_full_df = pd.concat(recon_dfs, ignore_index=True)
recon_full_df.to_csv(str(PROJECT_DIR / "results" / "sdc_real_key_reconciliation_doseresponse.csv"), index=False)

In [ ]:
conditions = [("toeplitz_only", {"toeplitz_prob": 0.5}), ("final_key_only", {"final_key_prob": 0.5})]
all_rows = []

for label, kwargs in conditions:
    print(f"\n=== {label} on real channel (asymptotic length) ===")
    for run in range(10):
        seed = 42 + run
        result = run_real_channel_trial(bob, alice, bob_ip, real_qber, run, seed, k=k_pe,
                                           length_mode="asymptotic", **kwargs)
        result["condition"] = label
        all_rows.append(result)
        print(f"  run {run}: keys_match={result.get('keys_match')}, "
              f"verification_passed={result.get('verification_passed')}, "
              f"secure_key_length={result.get('secure_key_length')}")

df = pd.DataFrame(all_rows)
df.to_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_toeplitz_finalkey_asymptotic.csv"), index=False)

In [ ]:
existing_df = pd.read_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_toeplitz_finalkey_asymptotic.csv"))
conditions = [("toeplitz_only", {"toeplitz_prob": 0.5}), ("final_key_only", {"final_key_prob": 0.5})]
new_rows = []

for label, kwargs in conditions:
    for run in range(10, 30):
        seed = 42 + run
        result = run_real_channel_trial(bob, alice, bob_ip, real_qber, run, seed, k=k_pe,
                                           length_mode="asymptotic", **kwargs)
        result["condition"] = label
        new_rows.append(result)

combined_df = pd.concat([existing_df, pd.DataFrame(new_rows)], ignore_index=True)
combined_df.to_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_toeplitz_finalkey_asymptotic.csv"), index=False)

In [ ]:
key_pairs_df = collect_key_pairs(deploy, slice_obj, alice, bob, bob_ip, PROJECT_DIR, n_keys=5)
# collect_key_pairs already saves this internally to key_pairs_metadata.csv
print(key_pairs_df)


In [9]:
#run if collecting new keys

alice.upload_file(str(PROJECT_DIR / "results" / "alice_sifted_bits_key0.json"),
                    "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(PROJECT_DIR / "results" / "alice_sifted_bits_key0.json"),
                  "qfabric/results/alice_sifted_bits.json")
bob.upload_file(str(PROJECT_DIR / "results" / "bob_sifted_bits_key0.json"),
                  "qfabric/results/bob_sifted_bits.json")

<SFTPAttributes: [ size=30687 uid=1000 gid=1000 mode=0o100644 atime=1787511428 mtime=1787512477 ]>

In [ ]:
# Fixed: use k AND qber from the SAME source (key_pairs_df's freshly-collected
# key0), rather than mixing k0 from key_pairs_df with real_qber from the
# original single split -- these can come from two different FABRIC
# collection rounds and may not actually match the same underlying key.

k0 = key_pairs_df.iloc[0]["k_pe"]
qber0 = key_pairs_df.iloc[0]["qber"]

new_rows = []
for prob in probs_to_run:
    print(f"\n=== reconciliation_prob={prob} ===")
    for run in range(10):
        seed = 42 + run
        result = run_real_channel_reconciliation_trial(bob, alice, bob_ip, qber0, run, seed,
                                                   reconciliation_prob=0.03, k=k0,
                                                   length_mode="asymptotic")
        new_rows.append(result)
        print(f"  run {run}: non_convergent={result.get('non_convergent')}, "
              f"faults_fired={result.get('faults_fired')}")

# Sanity check: flag it explicitly if this key0 differs from the original
# single-split k_pe/real_qber you've been using elsewhere in this notebook,
# rather than silently picking one and hoping they matched.
if abs(qber0 - real_qber) > 1e-6 or k0 != k_pe:
    print(f"WARNING: key_pairs_df key0 (k={k0}, qber={qber0:.4f}) differs from "
          f"the original single split (k={k_pe}, qber={real_qber:.4f}). "
          f"These are from separate FABRIC collection rounds -- using "
          f"key_pairs_df's own (k0, qber0) pair below, consistently.")
else:
    print(f"OK: key_pairs_df key0 matches the original single split (k={k0}, qber={qber0:.4f})")

recon_rows = []
for run in range(10):
    seed = 42 + run
    result = run_real_channel_reconciliation_trial(bob, alice, bob_ip, qber0, run, seed,
                                                   reconciliation_prob=0.03, k=k0,
                                                   length_mode="asymptotic")
    recon_rows.append(result)
    print(f"run {run}: non_convergent={result.get('non_convergent')}, "
          f"faults_fired={result.get('faults_fired')}")

recon_df = pd.DataFrame(recon_rows)
print(recon_df.to_string(index=False))
recon_df.to_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_reconciliation_key0_asymptotic.csv"), index=False)

In [ ]:
key_files = [(f"alice_sifted_bits_key{i}.json", f"bob_sifted_bits_key{i}.json") for i in range(5)]
conditions = [("toeplitz_only", {"toeplitz_prob": 0.5}), ("final_key_only", {"final_key_prob": 0.5})]

all_rows = []
for key_idx, (a_file, b_file) in enumerate(key_files):
    meta_row = key_pairs_df.iloc[key_idx]
    k_i, qber_i = int(meta_row["k_pe"]), float(meta_row["qber"])

    alice.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / b_file), "qfabric/results/bob_sifted_bits.json")

    print(f"\n=== key{key_idx}: {int(meta_row['n_bits'])} generation bits, k={k_i} PE sample, "
          f"QBER (PE sample) = {qber_i:.4f} (asymptotic length) ===")

    for label, kwargs in conditions:
        print(f"  Running {label}...")
        for run in range(5):
            seed = 42 + run
            result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, k=k_i,
                                               length_mode="asymptotic", **kwargs)
            result["condition"] = label
            result["key_index"] = key_idx
            result["qber"] = qber_i
            all_rows.append(result)
            print(f"    run {run}: keys_match={result.get('keys_match')}, "
                  f"faults_fired={result.get('faults_fired')}")

df_multikey = pd.DataFrame(all_rows)
df_multikey.to_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_toeplitz_finalkey_multikey_asymptotic.csv"), index=False)
print(f"\nSaved {len(df_multikey)} rows.")

In [ ]:
key_files = [(f"alice_sifted_bits_key{i}.json", f"bob_sifted_bits_key{i}.json") for i in range(5)]
probs = [0.5, 0.3, 0.1, 0.05, 0.01]

all_rows = []
for key_idx, (a_file, b_file) in enumerate(key_files):
    meta_row = key_pairs_df.iloc[key_idx]
    k_i, qber_i = int(meta_row["k_pe"]), float(meta_row["qber"])

    alice.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / a_file), "qfabric/results/alice_sifted_bits.json")
    bob.upload_file(str(PROJECT_DIR / "results" / b_file), "qfabric/results/bob_sifted_bits.json")
    print(f"\n=== key{key_idx}: QBER (PE sample) = {qber_i:.4f} (asymptotic length) ===")

    for prob in probs:
        for fault_type, kwargs in [("toeplitz", {"toeplitz_prob": prob}), ("final_key", {"final_key_prob": prob})]:
            print(f"  {fault_type} prob={prob}...")
            for run in range(3):
                seed = 42 + run
                result = run_real_channel_trial(bob, alice, bob_ip, qber_i, run, seed, k=k_i,
                                                   length_mode="asymptotic", **kwargs)
                result["fault_type"] = fault_type
                result["prob"] = prob
                result["key_index"] = key_idx
                result["qber"] = qber_i
                all_rows.append(result)

df_doseresponse = pd.DataFrame(all_rows)
df_doseresponse.to_csv(str(PROJECT_DIR / "results" / "sdc_real_channel_toeplitz_finalkey_doseresponse_multikey_asymptotic.csv"), index=False)
print(f"\nSaved {len(df_doseresponse)} rows.")

In [ ]:
"""
Real-channel reconciliation dose-response (asymptotic length): fill out
the sweep to match the density of the Mock-based curve.
"""
import pandas as pd

existing_path = PROJECT_DIR / "results" / "sdc_real_channel_reconciliation_key0_asymptotic.csv"
existing_recon = pd.read_csv(str(existing_path)) if existing_path.exists() else pd.DataFrame()

already_have = set(existing_recon["reconciliation_prob"].unique()) if len(existing_recon) else set()
print(f"Already have data for: {already_have}")

target_probs = [0.3, 0.1, 0.03, 0.01, 0.003, 0.001]
probs_to_run = [p for p in target_probs if p not in already_have]
print(f"Running: {probs_to_run}")

k0 = key_pairs_df.iloc[0]["k_pe"]
qber0 = key_pairs_df.iloc[0]["qber"]  # matches k0's source, not real_qber

new_rows = []
for prob in probs_to_run:
    print(f"\n=== reconciliation_prob={prob} ===")
    for run in range(10):
        seed = 42 + run
        result = run_real_channel_reconciliation_trial(bob, alice, bob_ip, qber0, run, seed,
                                                   reconciliation_prob=0.03, k=k0,
                                                   length_mode="asymptotic")
        new_rows.append(result)
        print(f"  run {run}: non_convergent={result.get('non_convergent')}, "
              f"faults_fired={result.get('faults_fired')}")

new_df = pd.DataFrame(new_rows)
combined = pd.concat([existing_recon, new_df], ignore_index=True)
combined.to_csv(str(existing_path), index=False)
print(f"\nSaved {len(combined)} total rows -> {existing_path}")

summary = combined.groupby("reconciliation_prob")["non_convergent"].agg(
    ['mean', 'count']).rename(columns={'mean': 'non_convergent_rate'})
print(summary)